# PC-FNO3D — session GPU complète (service atom + mesures M2, M4, M5, M6)

Un seul **Exécution → Tout exécuter**, sur GPU (L4 ou A100). Ce notebook :

1. monte le Drive (valide l'autorisation Google quand elle s'affiche) ;
2. demande l'archive du code **à chaque fois** (`pcfno3d_code.zip`, construite sur ines-pc par
   `scripts/package_for_colab.py`) : jamais une vieille copie restée sur le Drive ;
3. construit le cache des jours de **test et de validation** ;
4. remet atom en ligne derrière le tunnel nommé (si le secret `TUNNEL_CRED_CONTENTS` existe) ;
5. lance les mesures, chacune dans un dossier **neuf** du Drive, sans rien écraser :
   M2 (calibration sur la validation, DSO6 sur le test), M4 (AN1), M5 (EOL1, parc de
   **démonstration**), M6 (rejeu local des instants de test au format du contrat) ;
6. publie vers atom.zelqane.com **seulement** si le secret `ATOM_PUBLISH_TOKEN` existe ;
7. garde le service en vie.

Une mesure qui échoue est signalée et n'arrête pas les suivantes ni le service. Aucun secret
n'est écrit dans ce notebook, sur le disque ou dans la sortie.

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Code (archive téléversée depuis ines-pc à chaque session)

In [ ]:
import json, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import files

MYDRIVE = Path("/content/drive/MyDrive")
CIBLE = Path("/content/PFE")
DEPOSEE = Path("/content/pcfno3d_code.zip")   # panneau Fichiers -> Importer (session)
if DEPOSEE.exists():
    donnees = DEPOSEE.read_bytes()
    print(f"archive deposee dans la session : {DEPOSEE}")
else:
    print("Choisis l'archive sur le PC (ou depose-la dans le panneau Fichiers) :")
    recu = files.upload()
    nom = next((n for n in recu if n.startswith("pcfno3d_code") and n.endswith(".zip")), None)
    if nom is None:
        raise FileNotFoundError(f"fichier recu : {list(recu)} — il faut pcfno3d_code.zip")
    donnees = recu[nom]
ARCHIVE = MYDRIVE / "pcfno3d_code.zip"
ARCHIVE.write_bytes(donnees)
if CIBLE.exists():
    shutil.rmtree(CIBLE)
CIBLE.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE) as zf:
    zf.extractall(CIBLE)
tampon = CIBLE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q fastapi uvicorn python-multipart httpx mcp==2.3.0 pandera 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]
import numpy
if int(numpy.__version__.split(".")[0]) < 2:
    raise RuntimeError(f"numpy {numpy.__version__} : Execution -> Deconnecter et supprimer "
                       "l'environnement d'execution, puis Tout executer.")

def lancer(commande: str) -> None:
    """Commande obligatoire : le notebook s'arrete si elle echoue."""
    ip = get_ipython()
    ip.system(commande)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        raise RuntimeError(f"echec (code {code}) de : {commande}")

BILAN = {}

JOURNAUX = MYDRIVE / "aneo_fno3d_runs" / "v7" / "journaux_session"
JOURNAUX.mkdir(parents=True, exist_ok=True)

def mesurer(nom: str, commande: str) -> int:
    """Mesure facultative : un echec est note dans BILAN, la suite continue. La sortie est
    affichee ET ecrite ligne a ligne dans un journal du Drive (survit a une deconnexion)."""
    t0 = time.time()
    journal = JOURNAUX / f"{time.strftime('%Y%m%dT%H%M%S')}_{nom}.log"
    with open(journal, "w", encoding="utf-8") as fj:
        proc = subprocess.Popen(commande, shell=True, cwd="/content/PFE", text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for ligne in proc.stdout:
            print(ligne, end="", flush=True)
            fj.write(ligne)
            fj.flush()
        code = proc.wait()
    BILAN[nom] = {"code": code, "duree_min": round((time.time() - t0) / 60, 1),
                  "journal": str(journal)}
    print(f"\n>>> {nom} : code {code} ({BILAN[nom]['duree_min']} min) - journal {journal}")
    return code

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("pret")

## 3. Cache des jours de test et de validation

In [ ]:
import pandas as pd

DATASET = MYDRIVE / "aneo_dataset_3d_v2"
CACHE = Path("/content/aneo_cache_v7")
split = json.loads((DATASET / "split_days.json").read_text())
voulus = list(split["test_days"]) + list(split.get("val_days", []))
deja = set(pd.read_csv(CACHE / "index.csv")["day"]) if (CACHE / "index.csv").exists() else set()
manque = [d for d in voulus if d not in deja]
if manque:
    print(f"construction du cache : {len(manque)} jour(s)")
    lancer(f"python -m pcfno3d -c colab build-cache --days {','.join(map(str, manque))}")
index = pd.read_csv(CACHE / "index.csv")
print(index["split"].value_counts().to_string())

## 4. atom en ligne (tunnel nommé) — sauté si le secret du tunnel est absent

In [ ]:
from google.colab import userdata

INFO = None
try:
    _ident = userdata.get("TUNNEL_CRED_CONTENTS")
except Exception:
    _ident = None
if not _ident:
    print("secret TUNNEL_CRED_CONTENTS absent ou non autorise : service non lance, "
          "les mesures continuent.")
else:
    lancer("python -m pcfno3d -c colab export-web --split test --stride 2")
    from service.colab import serve_public
    INFO = serve_public(config="colab", key_name="atom-worker", scope="read",
                        key_hash="ce7965c54284eb6d1a274119839a71d9785731609d04b17bd08fb6da7d2c30fd",
                        named_tunnel={"id": "e552f718-2478-4fe3-82b4-8d33a4b9eb6e",
                                      "hostname": "atom-origin.zelqane.com",
                                      "credentials": _ident})
del _ident

In [ ]:
import urllib.request

if INFO is not None:
    req = urllib.request.Request("https://atom-origin.zelqane.com/api/health",
                                 headers={"User-Agent": "pcfno3d-colab/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            print(json.loads(r.read().decode()).get("status"))
    except Exception as exc:
        print("verification externe impossible :", type(exc).__name__)

## 5. M2 — calibration (validation) et DSO6 (test)

In [ ]:
mesurer("M2", "python scripts/m2_calibration_colab.py -c colab --deuxieme-lecture")
RUN = Path("/content/drive/MyDrive/aneo_fno3d_runs/v7")
def derniere(motif):
    c = sorted(RUN.glob(motif), key=lambda p: p.stat().st_mtime)
    return c[-1] if c else None
SESSION_M2 = derniere("fiabilite_pcfno3d/*/session_*")
print("session M2 :", SESSION_M2)
if SESSION_M2 and (SESSION_M2 / "resume.json").exists():
    print((SESSION_M2 / "resume.json").read_text())

## 6. M4 — AN1 (protocole fixe : verdict officiel)

In [ ]:
mesurer("M4", "python scripts/m4_an1_colab.py -c colab")
SESSION_M4 = derniere("alertes_pcfno3d/*/session_*")
print("session M4 :", SESSION_M4)
for f in ("an1_test.json", "resume.json"):
    if SESSION_M4 and (SESSION_M4 / f).exists():
        print(f, (SESSION_M4 / f).read_text()[:3000])

## 7. M5 — EOL1 sur le parc de DÉMONSTRATION (le verdict officiel attend le parc réel)

In [ ]:
mesurer("M5", "python scripts/m5_eol1_colab.py -c colab --parc configs/eolien/parc_demo.yaml")
SESSION_M5 = derniere("eolien_pcfno3d/*/eol1_*")
print("session M5 :", SESSION_M5)
if SESSION_M5:
    for f in sorted(SESSION_M5.glob("*.json"))[:3]:
        print(f.name, f.read_text()[:2000])

## 8. M6 — rejeu local des instants de test au format du contrat

In [ ]:
regl = []
cal = SESSION_M2 / "calibration_validation.json" if SESSION_M2 else None
if cal and cal.exists():
    regl += ["--set", f"inference.reliability={cal}"]
seuils = SESSION_M4 / "seuils_validation.json" if SESSION_M4 else None
if seuils and seuils.exists():
    regl += ["--set", "alerts.enabled=true", "--set", f"alerts.seuils={seuils}"]
regl += ["--set", "wind_farm.turbines=configs/eolien/parc_demo.yaml"]
dso6 = SESSION_M2 / "dso6.json" if SESSION_M2 else None
if dso6 and dso6.exists():
    regl += ["--dso6", str(dso6)]
ARGS_M6 = " ".join(f'"{a}"' for a in regl)
mesurer("M6-local", f"python scripts/m6_rejeu_colab.py -c colab --local --data-kind synthetique {ARGS_M6}")

## 8 bis. M8 — preuves (références log et krigeage, E1, E2, E3, E5)

In [ ]:
mesurer("M8", "python scripts/m8_preuves_colab.py -c colab")
SESSION_M8 = derniere("preuves_pcfno3d/*/session_*")
print("session M8 :", SESSION_M8)
if SESSION_M8 and (SESSION_M8 / "resume.json").exists():
    print((SESSION_M8 / "resume.json").read_text()[:4000])

## 9. Publication vers atom.zelqane.com — seulement si `ATOM_PUBLISH_TOKEN` existe

Le jeton vit dans les secrets Colab (icône clé), jamais dans le notebook. Le Worker R2 doit être
déployé avant (fait depuis ines-pc).

In [ ]:
import os
try:
    _jeton = userdata.get("ATOM_PUBLISH_TOKEN")
except Exception:
    _jeton = None
if not _jeton:
    print("pas de secret ATOM_PUBLISH_TOKEN : publication sautee (le rejeu local est fait).")
else:
    os.environ["ATOM_PUBLISH_TOKEN"] = _jeton
    mesurer("M6-publication", f"python scripts/m6_rejeu_colab.py -c colab "
            f"--remote https://atom.zelqane.com --data-kind synthetique {ARGS_M6}")
    os.environ.pop("ATOM_PUBLISH_TOKEN", None)
del _jeton

## 10. Bilan, puis garder le service en vie

In [ ]:
print(json.dumps(BILAN, indent=1, ensure_ascii=False))
(RUN / f"bilan_session_{time.strftime('%Y%m%dT%H%M%S')}.json").write_text(
    json.dumps(BILAN, indent=1, ensure_ascii=False))

In [ ]:
if INFO is not None:
    try:
        while True:
            t_ok = INFO["tunnel"].alive
            s_ok = INFO["process"].poll() is None
            print(time.strftime("%H:%M"), "| tunnel", "ok" if t_ok else "ARRETE",
                  "| service", "ok" if s_ok else "ARRETE", flush=True)
            if not (t_ok and s_ok):
                break
            time.sleep(600)
    except KeyboardInterrupt:
        print("surveillance arretee — le service tourne toujours")